ライブラリのインポートとグラフのフォント設定

In [ ]:
import os

import pymc as pm
import numpy as np
import pandas as pd
import arviz as az
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.preprocessing import StandardScaler
import tkinter as tk
from tkinter import filedialog
import warnings

# 警告を非表示にする設定
warnings.simplefilter(action='ignore', category=FutureWarning)

# --- フォント設定 ---
# 指定されたフォントファイルのパス
font_file_path = r"C:\Users\0uh2j\OneDrive\Desktop\IPAexfont00401\ipaexg.ttf"

# フォントが存在するか確認して適用
if os.path.exists(font_file_path):
    try:
        # Matplotlibのフォントマネージャーに追加
        fm.fontManager.addfont(font_file_path)
        # フォントプロパティを取得して名前を特定
        font_prop = fm.FontProperties(fname=font_file_path)
        font_name = font_prop.get_name()
        # デフォルトフォントとして設定
        plt.rcParams['font.family'] = font_name
        print(f"フォント '{font_name}' を適用しました。")
    except Exception as e:
        print(f"フォントの設定中にエラーが発生しました: {e}")
        print("デフォルトのフォントを使用します。")
else:
    print(f"指定されたフォントファイルが見つかりません: {font_file_path}")
    print("デフォルトのフォントを使用します。")

フォント 'IPAexGothic' を適用しました。


学習データファイル選択

In [2]:
root = tk.Tk()
root.withdraw()
print("分析するCSVファイルを選択してください...")
csv_file_path = filedialog.askopenfilename(
    title="CSVファイルを選択してください",
    filetypes=[("CSVファイル", "*.csv"), ("すべてのファイル", "*.*")]
)

分析するCSVファイルを選択してください...


学習データの読み込みと各変数の定義

学習の前処理(温度のケルビン換算、データの標準化とデータの中央化)

In [3]:
# 実行制御用のフラグ初期化
data_loaded = False

if csv_file_path:
    try:
        df = pd.read_csv(csv_file_path)
        
        # 必要な列名の定義
        col_temp = '流動中樹脂温度T3(℃)'
        col_pres = '流動中樹脂圧力P23(Pa)'
        col_sr = 'せん断速度γ23(1/s)'
        col_visc = '樹脂粘度η23(Pa・s)'
        
        required_columns = [col_temp, col_pres, col_sr, col_visc]
        if not all(col in df.columns for col in required_columns):
            raise ValueError(f"CSVファイルに必要な列が含まれていません。\n必要な列: {required_columns}")

        # 温度の単位変換 (℃ -> K)
        df['流動中樹脂温度T3(K)'] = df[col_temp] + 273.15
        
        # 対数変換
        log_T = np.log(df['流動中樹脂温度T3(K)'].values)
        log_P = np.log(df[col_pres].values)
        log_SR = np.log(df[col_sr].values)
        log_VISC = np.log(df[col_visc].values)
        
        # 入力データ X の作成と標準化
        X_raw = np.stack([log_T, log_P, log_SR], axis=1)
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X_raw)
        
        # 目的変数 y の中心化
        y_mean = log_VISC.mean()
        y_centered = log_VISC - y_mean
        
        data_loaded = True
        print(f"データを読み込み、{len(X_scaled)}件の学習準備が完了しました。")
        
    except Exception as e:
        print(f"エラーが発生しました: {e}")

データを読み込み、1076件の学習準備が完了しました。


ガウス過程回帰モデルの定義(ガウス過程、カーネル関数の定義)

In [4]:
if data_loaded:
    print("モデルを構築しています...")
    with pm.Model() as gp_visc_model:
        # --- ハイパーパラメータの定義 ---
        # eta: 関数の振幅
        eta = pm.HalfNormal("eta", sigma=2.0)
        # ls: 長さスケール (各入力次元に対する感度)
        ls = pm.Gamma("ls", alpha=2.0, beta=1.0, shape=3)
        # sigma: 観測ノイズ
        sigma = pm.HalfNormal("sigma", sigma=1.0)
        
        # --- カーネル関数の定義 (ARD付きガウスカーネル) ---
        cov_func = eta**2 * pm.gp.cov.ExpQuad(input_dim=3, ls=ls)
        
        # --- ガウス過程の定義 (Latent型) ---
        gp = pm.gp.Latent(cov_func=cov_func)
        
        # 潜在関数 f の事前分布
        f = gp.prior("f", X=X_scaled)
        
        # 尤度 (観測モデル)
        y_obs = pm.Normal("y_obs", mu=f, sigma=sigma, observed=y_centered)
        
    print("GPRモデルの定義が完了しました。")

モデルを構築しています...
GPRモデルの定義が完了しました。


計算アルゴリズム(マルコフ連鎖モンテカルロ法)の実行

In [ ]:
# 修正案（セル5の中身）
if data_loaded:
    with gp_visc_model:
        print("\n--- MCMCサンプリング開始 (GPR) ---")
        # progressbar=True を明示的に追加
        trace = pm.sample(
            1000, 
            tune=1000, 
            chains=4, 
            target_accept=0.99, 
            random_seed=42,
            progressbar=True  # ここを確認
        )
        print("--- サンプリング完了 ---")
        mcmc_completed = True


--- MCMCサンプリング開始 (GPR) ---


c:\Users\0uh2j\anaconda3\envs\viscosity_analysis_0120\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)
Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [eta, ls, sigma, f_rotated_]


Output()

学習結果の可視化

In [ ]:
if mcmc_completed:
    print("\n--- 学習結果の可視化を開始します ---")
    
    # プロットの設定
    input_vars = ['流動中樹脂温度T3', '流動中樹脂圧力P23', 'せん断速度γ23']
    input_units = ['(℃)', '(Pa)', '(1/s)']
    
    # 3つのグラフを横に並べる
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # 予測曲線を滑らかにするためのグリッド数
    n_grid = 100

    # モデルコンテキスト内で予測を実行
    with gp_visc_model:
        for i, (var_name, unit) in enumerate(zip(input_vars, input_units)):
            ax = axes[i]
            
            # --- 1. 予測用データの作成 (X_new) ---
            # 対象変数の最小値～最大値の範囲を取得
            x_min, x_max = X_scaled[:, i].min(), X_scaled[:, i].max()
            padding = (x_max - x_min) * 0.1
            x_grid = np.linspace(x_min - padding, x_max + padding, n_grid)
            
            # 3次元の入力データを作成 (対象変数以外は0=平均値に固定)
            X_new_scaled = np.zeros((n_grid, 3))
            X_new_scaled[:, i] = x_grid
            
            # --- 2. 事後予測分布の取得 ---
            f_star = gp.conditional(f"f_star_{i}", Xnew=X_new_scaled)
            # 予測サンプリングを実行
            ppc = pm.sample_posterior_predictive(trace, var_names=[f"f_star_{i}"], random_seed=42, progressbar=False)
            # サンプルデータを整形
            f_samples = ppc.posterior_predictive[f"f_star_{i}"].values.reshape(-1, n_grid)
            
            # --- 3. データの逆変換 ---
            # 予測値 (y): 中心化解除 -> exp
            y_pred_samples = f_samples + y_mean
            visc_pred_samples = np.exp(y_pred_samples)
            
            # 統計量
            visc_mean = visc_pred_samples.mean(axis=0)
            visc_hdi = az.hdi(visc_pred_samples, hdi_prob=0.94)
            
            # 横軸 (x): 標準化解除 -> exp -> (温度なら℃へ)
            x_grid_unscaled = x_grid * scaler.scale_[i] + scaler.mean_[i]
            x_grid_real = np.exp(x_grid_unscaled)
            
            # 学習データ (散布図用)
            x_train_real = np.exp(X_raw[:, i])
            y_train_real = np.exp(y_centered + y_mean)
            
            if i == 0: # 温度の場合 (K -> ℃)
                x_grid_real = x_grid_real - 273.15
                x_train_real = x_train_real - 273.15
            
            # --- 4. グラフ描画 ---
            # 学習データのプロット
            ax.scatter(x_train_real, y_train_real, color='black', alpha=0.3, s=15, label='学習データ')
            
            # 事後平均のプロット
            ax.plot(x_grid_real, visc_mean, color='red', linewidth=2, label='事後平均')
            
            # 94% HDI のプロット
            ax.fill_between(x_grid_real, visc_hdi[:, 0], visc_hdi[:, 1], color='red', alpha=0.2, label='94% HDI')
            
            # タイトルと軸ラベル
            ax.set_title(f"依存性プロット: {var_name}")
            ax.set_xlabel(f"{var_name} {unit}")
            ax.set_ylabel("樹脂粘度 (Pa・s)")
            
            # --- 軸の対数設定 ---
            # 縦軸（樹脂粘度）は常にログスケール
            ax.set_yscale('log')
            
            # 横軸：せん断速度の場合のみログスケール
            # (i=2 がせん断速度に対応)
            if i == 2: 
                ax.set_xscale('log')

            # グリッド (対数軸の方が見やすいよう 'both' と 'minor' を設定)
            ax.grid(True, which="both", linestyle='--', alpha=0.5)
            
            if i == 0:
                ax.legend()

    plt.tight_layout()
    plt.show()
    print("グラフを表示しました。")